<a href="https://colab.research.google.com/github/buvana-seshathri/CS6320_Assignment1/blob/main/encoder_decoder_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time

import numpy as np
import tensorflow as tf

In [2]:
filepath = tf.keras.utils.get_file("shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",)

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
text = open(filepath, "rb").read().decode(encoding="utf-8")
print(f"Length of text: {len(text)} characters")

Length of text: 1115394 characters


In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
vocab = sorted(set(text))
print("unique character count: ", len(vocab))

unique character count:  65


In [6]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None
)

In [7]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None
)

In [8]:
def text_from_ids(ids):
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

In [9]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, "UTF-8"))
all_ids

<tf.Tensor: shape=(1115394,), dtype=int64, numpy=array([19, 48, 57, ..., 46,  9,  1])>

In [10]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [11]:
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode("utf-8"))

F
i
r
s
t
 
C
i
t
i


In [12]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

In [13]:
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

for seq in sequences.take(2):
    print(chars_from_ids(seq))

tf.Tensor(
[b'F' b'i' b'r' b's' b't' b' ' b'C' b'i' b't' b'i' b'z' b'e' b'n' b':'
 b'\n' b'B' b'e' b'f' b'o' b'r' b'e' b' ' b'w' b'e' b' ' b'p' b'r' b'o'
 b'c' b'e' b'e' b'd' b' ' b'a' b'n' b'y' b' ' b'f' b'u' b'r' b't' b'h'
 b'e' b'r' b',' b' ' b'h' b'e' b'a' b'r' b' ' b'm' b'e' b' ' b's' b'p'
 b'e' b'a' b'k' b'.' b'\n' b'\n' b'A' b'l' b'l' b':' b'\n' b'S' b'p' b'e'
 b'a' b'k' b',' b' ' b's' b'p' b'e' b'a' b'k' b'.' b'\n' b'\n' b'F' b'i'
 b'r' b's' b't' b' ' b'C' b'i' b't' b'i' b'z' b'e' b'n' b':' b'\n' b'Y'
 b'o' b'u' b' '], shape=(101,), dtype=string)
tf.Tensor(
[b'a' b'r' b'e' b' ' b'a' b'l' b'l' b' ' b'r' b'e' b's' b'o' b'l' b'v'
 b'e' b'd' b' ' b'r' b'a' b't' b'h' b'e' b'r' b' ' b't' b'o' b' ' b'd'
 b'i' b'e' b' ' b't' b'h' b'a' b'n' b' ' b't' b'o' b' ' b'f' b'a' b'm'
 b'i' b's' b'h' b'?' b'\n' b'\n' b'A' b'l' b'l' b':' b'\n' b'R' b'e' b's'
 b'o' b'l' b'v' b'e' b'd' b'.' b' ' b'r' b'e' b's' b'o' b'l' b'v' b'e'
 b'd' b'.' b'\n' b'\n' b'F' b'i' b'r' b's' b't' b' ' b'C' b'i' b't' b'

In [14]:
for seq in sequences.take(5):
    print(text_from_ids(seq).numpy())

b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
b'are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you k'
b"now Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us ki"
b"ll him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be d"
b'one: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citi'


In [15]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [16]:
split_input_target(list("Tensorflow"))

(['T', 'e', 'n', 's', 'o', 'r', 'f', 'l', 'o'],
 ['e', 'n', 's', 'o', 'r', 'f', 'l', 'o', 'w'])

In [17]:
dataset = sequences.map(split_input_target)

In [18]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target: b'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


In [19]:
# Batch size
BATCH_SIZE = 64

BUFFER_SIZE = 10000

dataset = (
    dataset.shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE)
)

dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [20]:
# Length of the vocabulary in chars
vocab_size = len(vocab)

# The embedding dimension - size of vector we want to represent our characters with
embedding_dim = 256

# Number of RNN units - no of neurons
rnn_units = 1024

In [21]:
class MyModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        # TODO - Create an embedding layer
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        # TODO - Create a GRU layer
        self.gru = tf.keras.layers.GRU(
            rnn_units, return_sequences=True, return_state=True
        )
        # TODO - Finally connect it with a dense layer
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)
        # Pass states directly to the GRU layer. If states is None,
        # the GRU layer will initialize a zero state internally.
        x, states = self.gru(x, initial_state=states, training=training)
        x = self.dense(x, training=training)

        if return_state:
            return x, states
        else:
            return x

In [22]:
model = MyModel(
    # Be sure the vocabulary size matches the `StringLookup` layers.
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
)

In [23]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(
        example_batch_predictions.shape,
        "# (batch_size, sequence_length, vocab_size)",
    )

(64, 100, 66) # (batch_size, sequence_length, vocab_size)


In [24]:
model.summary()

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 100, 256)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ((64, 100, 1024), (64, │     3,938,304 │
│                                 │ 1024))                 │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 100, 66)          │        67,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,022,850 (15.35 MB)

 Trainable params: 4,022,850 (15.35 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
sampled_indices = tf.random.categorical(
    example_batch_predictions[0], num_samples=1
)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

In [26]:
sampled_indices

array([ 1,  4, 53,  9, 48, 49,  2, 65,  7, 25, 11, 20, 42, 45, 21, 15,  5,
        6, 40,  8, 60,  7,  9,  7, 13, 16,  3,  5, 31, 58, 46, 64, 10, 44,
       28, 11, 28, 27, 52,  5, 19, 41, 34, 53, 50, 48, 57, 17, 47,  4, 22,
       18, 18, 32, 50, 37, 25,  2, 24, 28, 21, 38,  4, 13, 10,  4, 10, 56,
       54, 36, 15, 24, 21, 23,  2, 27, 13, 53, 59, 63, 42,  0, 21, 49,  6,
       29, 28, 14, 39, 52, 38, 17,  2, 18, 45, 34, 20, 32,  6, 48])

In [27]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b"\n'Tis well; and I have met a gentleman\nHath promised me to help me to another,\nA fine musician to in"

Next Char Predictions:
 b"\n$n.ij z,L:GcfHB&'a-u,.,?C!&Rsgy3eO:ONm&FbUnkirDh$IEESkXL KOHY$?3$3qoWBKHJ N?ntxc[UNK]Hj'POAZmYD EfUGS'i"


In [28]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

In [29]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print(
    "Prediction shape: ",
    example_batch_predictions.shape,
    " # (batch_size, sequence_length, vocab_size)",
)
print("Mean loss:        ", example_batch_mean_loss)

Prediction shape:  (64, 100, 66)  # (batch_size, sequence_length, vocab_size)
Mean loss:         tf.Tensor(4.188641, shape=(), dtype=float32)


In [30]:
tf.exp(example_batch_mean_loss).numpy()

np.float32(65.933136)

In [31]:
model.compile(optimizer="adam", loss=loss)

In [32]:
# Directory where the checkpoints will be saved
checkpoint_dir = "./training_checkpoints"
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix, save_weights_only=True
)

In [38]:
EPOCHS = 30

In [39]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 1.1238
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 1.0813
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 1.0368
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.9906
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.9433
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.8934
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.8440
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.7956
Epoch 9/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.7488
Epoch 10/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.7045
Epoch 11/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.6644
Epoch 12/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.6269
Epoch 13/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.5958
Epoch 14/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.5680
Epoch 15/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 3s

In [40]:
class OneStep(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # Create a mask to prevent "[UNK]" from being generated.
        skip_ids = self.ids_from_chars(["[UNK]"])[:, None]
        sparse_mask = tf.SparseTensor(
            # Put a -inf at each bad index.
            values=[-float("inf")] * len(skip_ids),
            indices=skip_ids,
            # Match the shape to the vocabulary
            dense_shape=[len(ids_from_chars.get_vocabulary())],
        )
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convert strings to token IDs.
        input_chars = tf.strings.unicode_split(inputs, "UTF-8")
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # Run the model.
        # predicted_logits.shape is [batch, char, next_char_logits]
        predicted_logits, states = self.model(
            inputs=input_ids, states=states, return_state=True
        )
        # Only use the last prediction.
        predicted_logits = predicted_logits[:, -1, :]
        predicted_logits = predicted_logits / self.temperature
        # Apply the prediction mask: prevent "[UNK]" from being generated.
        predicted_logits = predicted_logits + self.prediction_mask

        # Sample the output logits to generate token IDs.
        predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convert from token ids to characters
        predicted_chars = self.chars_from_ids(predicted_ids)

        # Return the characters and model state.
        return predicted_chars, states

In [41]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

In [42]:
start = time.time()
states = None
next_char = tf.constant(["ROMEO:"])
result = [next_char]

for n in range(1000):
    next_char, states = one_step_model.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode("utf-8"), "\n\n" + "_" * 80)
print("\nRun time:", end - start)

ROMEO:
These four will mother should so pettity
'Tis so: for your better cannot brow
Then vail your ignorance, if there be drowning
That rebelo!' I were God befall'n;
And I would have stood to end,
Yet neat the act or't.

First Lady:
Clarence close, you show you are to make his pointed.

FRIAR LAURENCE:
Sir, here comes a man; and, going affecians bark absolute
As what within the king and thou didst beg.

Keepereor:
Let me entreat you.

BENVOLIO:
I pray thee, feeling.
Speak o' the fowl! why have care hence!
For God's sake, lady, oak not Marcius, and
What they have paid for landers lives.

LEONTES:
Shall need no more.

Third Citizen:
Before the foes that sees in murders for thee
And most of our immortal blessmend.

DERBY:
I cannot tell: there's not a smilerbath'd sour.

HASTINGS:
No inetory, sir: I have not dismish'd
To think it will. What sir, a sail! that
love he be authorely now
That would be glad to weed I stay, if they
account of women witchy woo.
Banished, how must he not so?
Our t

In [43]:
start = time.time()
states = None
next_char = tf.constant(["ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:"])
result = [next_char]

for n in range(1000):
    next_char, states = one_step_model.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

result = tf.strings.join(result)
end = time.time()
for s in result:
    print(s.numpy().decode("utf-8"))
print("\n\n" + "_" * 80)
print("\nRun time:", end - start)

ROMEO:
Thou fortune to them golden window then endorch his power.

WARWICK:
Oxford, how now! what's the neck?

Servant:
Up.

ROMEO:
Petruchio, go thy ways; though it esteen a
renown in rose and the law untitured lose.
What stay him I will not be king?

DUKE OF YORK:
My lords, let it o'er his comperent in
Vienna bent, good night! knee lift delights;
And cannot for his gracious uncle Clarence,
Till thou repossess the spector of his light,
Both forbids the Duke of Gluck Freyzy, at the loanth
Of both is base and lost alone.

Nurse:
May not I tell thee poison.

Third Watchman:
By this all, or I'll kill the honour of my long.

LADY ANNE:
How far is it, my lord, this very tide.
Come, my friends; we will undeed, get thee in them;
And shall the sacrifing Warwick for
the storment before that tond:
The fouler grace me from the earth and thy bed.

MIRANDA:
I pray now, change it the queen these are:
The city go in Gentleman, the physic

ATH:
Both mine, by Standal, free, royally require,
Young puyes

In [44]:
tf.saved_model.save(one_step_model, "one_step")
one_step_reloaded = tf.saved_model.load("one_step")

In [45]:
states = None
next_char = tf.constant(["ROMEO:"])
result = [next_char]

for n in range(100):
    next_char, states = one_step_reloaded.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))

ROMEO:
The king's, so barilling too much,
That his beloved, so begins to me;
And, grief, my daughter, will


In [46]:
class CustomTraining(MyModel):
    @tf.function
    def train_step(self, inputs):
        inputs, labels = inputs
        with tf.GradientTape() as tape:
            predictions = self(inputs, training=True)
            loss = self.loss(labels, predictions)
        grads = tape.gradient(loss, model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, model.trainable_variables))

        return {"loss": loss}

In [47]:
model = CustomTraining(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
)

In [48]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [49]:
model.fit(dataset, epochs=1)

172/172 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.0000e+00


In [53]:
EPOCHS = 30

mean = tf.keras.metrics.Mean()

for epoch in range(EPOCHS):
    start = time.time()

    mean.reset_states()
    for batch_n, (inp, target) in enumerate(dataset):
        logs = model.train_step([inp, target])
        mean.update_state(logs["loss"])

        if batch_n % 50 == 0:
            template = (
                f"Epoch {epoch+1} Batch {batch_n} Loss {logs['loss']:.4f}"
            )
            print(template)

    # saving (checkpoint) the model every 5 epochs
    if (epoch + 1) % 5 == 0:
        model.save_weights(checkpoint_prefix.format(epoch=epoch))

    print()
    print(f"Epoch {epoch+1} Loss: {mean.result().numpy():.4f}")
    print(f"Time taken for 1 epoch {time.time() - start:.2f} sec")
    print("_", 80)

model.save_weights(checkpoint_prefix.format(epoch=epoch))

Epoch 1 Batch 0 Loss 2.0081
Epoch 1 Batch 50 Loss 1.8900
Epoch 1 Batch 100 Loss 1.8108
Epoch 1 Batch 150 Loss 1.7125

Epoch 1 Loss: 1.8331
Time taken for 1 epoch 3.37 sec
_ 80
Epoch 2 Batch 0 Loss 1.6498
Epoch 2 Batch 50 Loss 1.6229
Epoch 2 Batch 100 Loss 1.6332
Epoch 2 Batch 150 Loss 1.5112

Epoch 2 Loss: 1.5903
Time taken for 1 epoch 2.85 sec
_ 80
Epoch 3 Batch 0 Loss 1.5208
Epoch 3 Batch 50 Loss 1.5005
Epoch 3 Batch 100 Loss 1.4458
Epoch 3 Batch 150 Loss 1.4308

Epoch 3 Loss: 1.4644
Time taken for 1 epoch 2.83 sec
_ 80
Epoch 4 Batch 0 Loss 1.4271
Epoch 4 Batch 50 Loss 1.4063
Epoch 4 Batch 100 Loss 1.3587
Epoch 4 Batch 150 Loss 1.4047

Epoch 4 Loss: 1.3882
Time taken for 1 epoch 2.85 sec
_ 80
Epoch 5 Batch 0 Loss 1.3310
Epoch 5 Batch 50 Loss 1.3140
Epoch 5 Batch 100 Loss 1.3004
Epoch 5 Batch 150 Loss 1.3304

Epoch 5 Loss: 1.3334
Time taken for 1 epoch 2.91 sec
_ 80
Epoch 6 Batch 0 Loss 1.2518
Epoch 6 Batch 50 Loss 1.2844
Epoch 6 Batch 100 Loss 1.3044
Epoch 6 Batch 150 Loss 1.2948

Ep

In [54]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

In [55]:
start = time.time()
states = None
next_char = tf.constant(["ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:"])
result = [next_char]

for n in range(1000):
    next_char, states = one_step_model.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

result = tf.strings.join(result)
end = time.time()
for s in result:
    print(s.numpy().decode("utf-8"))
print("\n\n" + "_" * 80)
print("\nRun time:", end - start)

ROMEO:
The underetorable rap with king?

GLOUCESTER:
Here both in full of winest;
Since if I see no fear? Now, keep your hate,
But as it were, having no more than mercy
I' the bill my followers in some bold,
Unto the world that cannot weed me at the other.
Thou hast been so, or she must bear?

MENENIUS:
Has he did call him for the gold, homicive,
The fresh creature and the benefit his deep
Sin enough: thou art a traitor
And said 'Ay he's your highness' say home:
These eyes shall reason the king;
But whether here see most of all, things corn,
I do peremptory I am no male.
This was my speech, and I must yield confess:
A messenger,
That couched every cross of those.

CAMILLO:
Ay, my good friends; which thou shalt heart.
Stand fast, he counts like one thing of
Plants on my hands, here in the villany.

GLOUCESTER:

KING EDWARD IV:
Why, so: now we be of his mother?

JOHN OF GAUNT:
God in her name, hath that, if he drew sleep.

ROMEO:
Lady, you go to? Hast thou not any welcome.

SICINIUS:
Hol